<a href="https://colab.research.google.com/github/dvarelaj/nlp-miniproyecto-icesi/blob/main/Entrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn seaborn

In [ ]:
pip install huggingface_hub

In [ ]:
#Librerias
from datasets import load_dataset
import pandas as pd
from huggingface_hub import HfApi
import os
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import json
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.preprocessing import LabelEncoder
import json, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from datasets import Dataset, DatasetDict
from transformers import (
    pipeline, AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
import evaluate
import torch

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 0 if torch.cuda.is_available() else -1
print("GPU disponible ✅" if device == 0 else "Solo CPU ⚠️")

TEXT_COL  = "raw"
LABEL_COL = "document_type"


##Importar dataset

In [ ]:
token = os.environ.get("HF_TOKEN")
if token:
    api = HfApi()
    print("Whoami:", api.whoami())

In [30]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import HfApi
api = HfApi()
print("Autenticado como:", api.whoami()["name"])

Autenticado como: DianaCVarela


In [ ]:
from datasets import load_dataset

ds = load_dataset("Meddies/meddies-pii-cleaned-v1", "spanish")
df_train = ds['train'].to_pandas()

In [ ]:
# Codificamos document_type como entero
le = LabelEncoder()
df_train["label"] = le.fit_transform(df_train[LABEL_COL])

id2label = {i: l for i, l in enumerate(le.classes_)}
label2id = {l: i for i, l in id2label.items()}
LABEL_NAMES = list(le.classes_)

print(f"Clases ({len(LABEL_NAMES)}):\n{LABEL_NAMES}")
print(f"\nDistribución:\n{df_train[LABEL_COL].value_counts()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribución de clases
counts = df_train[LABEL_COL].value_counts()
axes[0].barh(counts.index, counts.values, color='steelblue', edgecolor='white')
axes[0].set_title("Distribución de tipos de documento", fontweight='bold')
axes[0].set_xlabel("Cantidad de muestras")
for i, v in enumerate(counts.values):
    axes[0].text(v + 20, i, str(v), va='center', fontsize=8)

# Longitud del texto por clase
df_train["text_len"] = df_train[TEXT_COL].astype(str).apply(len)
medians = df_train.groupby(LABEL_COL)["text_len"].median().sort_values()
axes[1].barh(medians.index, medians.values, color='coral', edgecolor='white')
axes[1].set_title("Longitud mediana del texto por tipo (caracteres)", fontweight='bold')
axes[1].set_xlabel("Caracteres")

plt.suptitle("EDA – Documentos médicos en español (Meddies PII)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("eda.png", bbox_inches='tight')
plt.show()

print(f"Longitud promedio : {df_train['text_len'].mean():.0f} caracteres")
print(f"Máxima: {df_train['text_len'].max()} | Mínima: {df_train['text_len'].min()}")

In [ ]:
# Tomamos 6.400 ejemplos estratificados (400 por clase × 16)
# Esto garantiza reproducibilidad y velocidad en Colab T4
df_sample, _ = train_test_split(
    df_train, train_size=8000,
    stratify=df_train["label"], random_state=SEED
)

df_tr, df_temp = train_test_split(
    df_sample, test_size=0.2, stratify=df_sample["label"], random_state=SEED
)
df_val, df_te = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp["label"], random_state=SEED
)

print(f"Train: {len(df_tr)} | Val: {len(df_val)} | Test: {len(df_te)}")

def make_hf_dataset(df):
    return Dataset.from_dict({
        "text": df[TEXT_COL].astype(str).tolist(),
        "label": df["label"].tolist()
    })

raw_datasets = DatasetDict({
    "train":      make_hf_dataset(df_tr),
    "validation": make_hf_dataset(df_val),
    "test":       make_hf_dataset(df_te)
})

print(raw_datasets)

#Técnica 1: Zero-Shot

In [ ]:
# ── Zero-Shot con BART-MNLI ──────────────────────────────────────────────────
# No necesita entrenamiento. Le pasamos los nombres de clase directamente.
# Usamos las etiquetas en inglés porque BART-MNLI fue entrenado en inglés.

zs = pipeline("zero-shot-classification",
               model="facebook/bart-large-mnli",
               device=device)

# Etiquetas descriptivas en inglés para el modelo zero-shot
ZS_LABELS = [
    "medical bill", "immunization record", "medication chart",
    "prescription", "physical therapy plan", "consent form",
    "mortality report", "medical record", "lab result",
    "surgery schedule", "outpatient examination", "nursing note",
    "imaging report", "discharge summary", "transfer form",
    "referral letter"
]

# Evaluamos en el test set completo (batch para velocidad)
print("Evaluando Zero-Shot... (puede tardar 3-5 min en T4)")

texts_test  = df_te[TEXT_COL].astype(str).tolist()
labels_test = df_te["label"].tolist()

# Truncamos a 512 caracteres para no saturar el modelo
texts_truncated = [t[:512] for t in texts_test]

zs_preds_str = []
batch_size = 8
for i in range(0, len(texts_truncated), batch_size):
    batch = texts_truncated[i:i+batch_size]
    results = zs(batch, ZS_LABELS, batch_size=batch_size)
    if isinstance(results, dict):
        results = [results]
    for r in results:
        zs_preds_str.append(r["labels"][0])
    if i % 80 == 0:
        print(f"  {min(i+batch_size, len(texts_truncated))}/{len(texts_truncated)}")

# Mapeamos las predicciones de texto a entero
# Construimos el mapeo: "nursing note" → "NURSING_NOTE"
zs_label_map = {zl: cl for zl, cl in zip(ZS_LABELS, LABEL_NAMES)}
zs_preds = [label2id.get(zs_label_map.get(p, LABEL_NAMES[0]), 0) for p in zs_preds_str]

# Métricas
zs_acc = accuracy_score(labels_test, zs_preds)
zs_f1  = f1_score(labels_test, zs_preds, average='weighted', zero_division=0)

print(f"\n📊 Zero-Shot → Accuracy: {zs_acc:.4f} | F1 weighted: {zs_f1:.4f}")
print(classification_report(labels_test, zs_preds, target_names=LABEL_NAMES, zero_division=0))

# Guardamos para la tabla final
resultados = {"Zero-Shot (BART)": {"accuracy": zs_acc, "f1": zs_f1}}

In [ ]:
# ── Técnica 1b: Zero-Shot con BETO-XNLI (español nativo) ────────────────────
# Recognai/bert-base-spanish-wwm-cased-xnli es BETO fine-tuneado en XNLI
# (inferencia de lenguaje natural en español). Mucho más apropiado que BART
# para textos en español, y más liviano (~420MB vs 1.6GB de BART).

zs_beto = pipeline(
    "zero-shot-classification",
    model="Recognai/bert-base-spanish-wwm-cased-xnli",
    device=device
)

# Etiquetas en español — coherente con el idioma del modelo y del dataset
ZS_LABELS_ES = [
    "factura médica",
    "registro de vacunación",
    "gráfica de medicamentos",
    "prescripción médica",
    "plan de fisioterapia",
    "formulario de consentimiento",
    "informe de mortalidad",
    "expediente médico",
    "resultado de laboratorio",
    "agenda quirúrgica",
    "examen ambulatorio",
    "nota de enfermería",
    "informe de imagen",
    "resumen de alta",
    "formulario de traslado",
    "carta de derivación"
]

# Mapeo: etiqueta en español → código interno del dataset
zs_map_es = {zl: cl for zl, cl in zip(ZS_LABELS_ES, LABEL_NAMES)}

texts_test  = df_te[TEXT_COL].astype(str).tolist()
labels_test = df_te["label"].tolist()
texts_truncated = [t[:512] for t in texts_test]

print("Evaluando BETO-XNLI Zero-Shot...")
preds_beto_zs_str = []
batch_size = 8

for i in range(0, len(texts_truncated), batch_size):
    batch = texts_truncated[i:i+batch_size]
    results = zs_beto(batch, ZS_LABELS_ES, batch_size=batch_size)
    if isinstance(results, dict):
        results = [results]
    for r in results:
        preds_beto_zs_str.append(r["labels"][0])
    if i % 160 == 0:
        print(f"  {min(i+batch_size, len(texts_truncated))}/{len(texts_truncated)}")

preds_beto_zs = [label2id.get(zs_map_es.get(p, LABEL_NAMES[0]), 0)
                 for p in preds_beto_zs_str]

acc_beto_zs = accuracy_score(labels_test, preds_beto_zs)
f1_beto_zs  = f1_score(labels_test, preds_beto_zs, average='weighted', zero_division=0)

print(f"\n📊 BETO-XNLI Zero-Shot → Accuracy: {acc_beto_zs:.4f} | F1 weighted: {f1_beto_zs:.4f}")
print(classification_report(labels_test, preds_beto_zs, target_names=LABEL_NAMES, zero_division=0))

# Actualizamos el diccionario de resultados
resultados = {
    "Zero-Shot BART (inglés)":    {"accuracy": 0.0600, "f1": 0.0262},  # resultado anterior
    "Zero-Shot BETO-XNLI (español)": {"accuracy": acc_beto_zs, "f1": f1_beto_zs},
}

print("\n📋 Comparación Zero-Shot:")
for nombre, m in resultados.items():
    print(f"  {nombre:42s}  Acc: {m['accuracy']:.4f}  F1: {m['f1']:.4f}")

## Técnica 1 – Zero-Shot Classification

Se evaluaron dos modelos zero-shot como baseline, sin ningún entrenamiento
sobre el dataset:

| Modelo | Idioma | Accuracy | F1 weighted |
|--------|--------|----------|-------------|
| `facebook/bart-large-mnli` | Inglés | 0.0600 | 0.0262 |
| `Recognai/bert-base-spanish-wwm-cased-xnli` | Español | 0.0500 | 0.0378 |

### Análisis

Ambos modelos obtuvieron resultados cercanos al azar (1/16 clases = 6.25% esperado).
Esto no es un fallo del enfoque zero-shot en general, sino una consecuencia directa
de las características del dataset:

- Las 16 clases son documentos del mismo dominio (médico), con vocabulario muy solapado.
  Sin ejemplos de entrenamiento, ningún modelo puede distinguir entre una `NURSING_NOTE`
  y un `MEDICAL_RECORD` basándose solo en hipótesis genéricas.

- BART (inglés) concentró sus predicciones en `MEDICATION_CHART` (recall 0.76),
  lo que explica su accuracy levemente superior pero F1 inferior: no aprendió nada,
  simplemente apostó siempre a una clase.

- BETO-XNLI (español) distribuyó mejor sus predicciones, reflejado en un F1 más alto
  (0.0378 vs 0.0262), aunque su accuracy sea marginalmente menor.

**Conclusión:** El zero-shot establece un techo muy bajo (~6%) que el fine-tuning
deberá superar. Este resultado motiva directamente el uso de modelos ajustados
al dataset en las siguientes secciones.

#Fine-tuning BETO

## Tokenización

In [ ]:
# ── Técnica 2: Fine-tuning BETO ──────────────────────────────────────────────
# dccuchile/bert-base-spanish-wwm-cased es BETO: BERT entrenado desde cero
# en texto español (Wikipedia + corpus legal/news). A diferencia del BETO-XNLI
# que usamos antes (solo inferencia), aquí ajustamos TODOS los pesos del modelo
# a nuestro dataset de documentos médicos.

BETO_MODEL = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer_beto = AutoTokenizer.from_pretrained(BETO_MODEL)

def tokenizar(examples):
    return tokenizer_beto(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256  # documentos médicos son más largos que tweets
    )

tokenized_beto = raw_datasets.map(tokenizar, batched=True)
tokenized_beto = tokenized_beto.rename_column("label", "labels")
tokenized_beto.set_format("torch")

print("Dataset tokenizado:")
print(tokenized_beto)
print(f"\nEjemplo de tokens (primeros 10): {tokenized_beto['train'][0]['input_ids'][:10]}")

##Entrenamiento

In [ ]:
# Función de métricas
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1  = f1_metric.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1_weighted": f1["f1"]}

# Modelo con cabeza de clasificación para 16 clases
model_beto = AutoModelForSequenceClassification.from_pretrained(
    BETO_MODEL,
    num_labels=16,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

total_params = sum(p.numel() for p in model_beto.parameters())
print(f"Parámetros totales BETO: {total_params:,}")

# Argumentos de entrenamiento
training_args_beto = TrainingArguments(
    output_dir="./beto_medico",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",        # ← cambio aquí
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=30,
    seed=SEED,
    report_to="none"
)

trainer_beto = Trainer(
    model=model_beto,
    args=training_args_beto,
    train_dataset=tokenized_beto["train"],
    eval_dataset=tokenized_beto["validation"],
    compute_metrics=compute_metrics,
)

print("🚀 Entrenando BETO... (aprox. 10-15 min en Colab T4)")
trainer_beto.train()
print("✅ Entrenamiento completo")

##Evaluación y curva de entrenamiento

In [ ]:
# ── Evaluación en test set ───────────────────────────────────────────────────
preds_beto_out = trainer_beto.predict(tokenized_beto["test"])
preds_beto     = np.argmax(preds_beto_out.predictions, axis=-1)
labels_test    = preds_beto_out.label_ids

acc_beto = accuracy_score(labels_test, preds_beto)
f1_beto  = f1_score(labels_test, preds_beto, average="weighted", zero_division=0)

print(f"📊 BETO Fine-tuning → Accuracy: {acc_beto:.4f} | F1 weighted: {f1_beto:.4f}")
print(classification_report(labels_test, preds_beto, target_names=LABEL_NAMES, zero_division=0))

# Guardamos para la tabla final
resultados["Fine-tuning BETO"] = {"accuracy": acc_beto, "f1": f1_beto}

# ── Curva de entrenamiento ───────────────────────────────────────────────────
log_history = trainer_beto.state.log_history

train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs  = [l for l in log_history if "eval_loss" in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot([l["step"] for l in train_logs],
             [l["loss"] for l in train_logs],
             color="steelblue", alpha=0.7, label="Train loss")
axes[0].scatter([l["step"] for l in eval_logs],
                [l["eval_loss"] for l in eval_logs],
                color="tomato", s=80, zorder=5, label="Val loss")
axes[0].plot([l["step"] for l in eval_logs],
             [l["eval_loss"] for l in eval_logs],
             color="tomato", linestyle="--", alpha=0.6)
axes[0].set_title("Curva de pérdida – BETO", fontweight="bold")
axes[0].set_xlabel("Pasos")
axes[0].set_ylabel("Loss")
axes[0].legend()

# F1
axes[1].plot([l["epoch"] for l in eval_logs],
             [l["eval_f1_weighted"] for l in eval_logs],
             color="seagreen", marker="o", linewidth=2)
axes[1].set_title("F1 weighted por época – BETO", fontweight="bold")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("F1 weighted")
axes[1].set_ylim(0, 1)
for l in eval_logs:
    axes[1].annotate(f"{l['eval_f1_weighted']:.3f}",
                     (l["epoch"], l["eval_f1_weighted"]),
                     textcoords="offset points", xytext=(0, 8), fontsize=9)

plt.suptitle("Entrenamiento BETO – Documentos médicos en español", fontweight="bold")
plt.tight_layout()
plt.savefig("beto_training.png", bbox_inches="tight")
plt.show()

# ── Matriz de confusión ──────────────────────────────────────────────────────
cm = confusion_matrix(labels_test, preds_beto)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_title("Matriz de confusión – BETO Fine-tuning", fontweight="bold", fontsize=13)
ax.set_ylabel("Etiqueta real")
ax.set_xlabel("Etiqueta predicha")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("beto_confusion.png", bbox_inches="tight")
plt.show()

## Análisis – Fine-tuning BETO

**Accuracy: 80.63% | F1 weighted: 0.8086**

El fine-tuning de BETO sobre el dataset médico produce una mejora de ~75 puntos
porcentuales respecto al baseline zero-shot (6%), confirmando que el entrenamiento
supervisado es indispensable para dominios especializados.

### Hallazgos por clase

**Clases bien clasificadas (F1 > 0.85):**
`CONSENT_FORM` (0.95), `LAB_RESULT` (0.87), `MEDICAL_BILL` (0.87),
`MORTALITY_REPORT` (0.89), `TRANSFER_FORM` (0.87). Estas clases tienen
vocabulario estructurado y distintivo — formularios legales, valores numéricos
de laboratorio, datos de facturación — que el modelo aprende a reconocer fácilmente.

**Clases problemáticas (F1 < 0.75):**
`MEDICAL_RECORD` (0.63) y `OUTPATIENT_EXAMINATION` (0.60) son las más difíciles.
La matriz de confusión muestra que `MEDICAL_RECORD` se confunde frecuentemente
con `OUTPATIENT_EXAMINATION` (10 casos) y `DISCHARGE_SUMMARY` (4 casos). Esto es
semánticamente coherente: los tres son documentos narrativos del historial clínico
del paciente con vocabulario muy solapado.

### Sobre las curvas de entrenamiento

El modelo convergió alrededor de la época 3 (F1: 0.810), con una ganancia mínima
en la época 4 (F1: 0.811). La brecha pequeña entre Train Loss y Val Loss descarta
overfitting. Con más épocas, el modelo probablemente no mejoraría sustancialmente.

#Fine-tuning XLM-RoBERTa

##Tokenización

In [ ]:
# ── Técnica 3: Fine-tuning XLM-RoBERTa ───────────────────────────────────────
# xlm-roberta-base es un modelo multilingüe entrenado en 100 idiomas incluyendo
# español. A diferencia de BETO (solo español), XLM-RoBERTa aprendió
# representaciones compartidas entre idiomas, lo que puede ayudar o perjudicar
# en dominios muy específicos. Aquí lo comparamos directamente contra BETO
# en las mismas condiciones de entrenamiento.

XLM_MODEL = "xlm-roberta-base"

tokenizer_xlm = AutoTokenizer.from_pretrained(XLM_MODEL)

def tokenizar_xlm(examples):
    return tokenizer_xlm(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

# Reutilizamos raw_datasets (mismo split que BETO para comparación justa)
tokenized_xlm = raw_datasets.map(tokenizar_xlm, batched=True)
tokenized_xlm = tokenized_xlm.rename_column("label", "labels")
tokenized_xlm.set_format("torch")

print("Dataset tokenizado para XLM-RoBERTa:")
print(tokenized_xlm)

##Entrenamiento

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model_xlm = AutoModelForSequenceClassification.from_pretrained(
    XLM_MODEL,
    num_labels=16,
    id2label=id2label,
    label2id=label2id,
)

total_params = sum(p.numel() for p in model_xlm.parameters())
print(f"Parámetros totales XLM-RoBERTa: {total_params:,}")

# Mismos hiperparámetros que BETO para comparación justa
training_args_xlm = TrainingArguments(
    output_dir="./xlm_medico",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=30,
    seed=SEED,
    report_to="none"
)

trainer_xlm = Trainer(
    model=model_xlm,
    args=training_args_xlm,
    train_dataset=tokenized_xlm["train"],
    eval_dataset=tokenized_xlm["validation"],
    compute_metrics=compute_metrics,
)

print("🚀 Entrenando XLM-RoBERTa... (aprox. 10-15 min en Colab T4)")
trainer_xlm.train()
print("✅ Entrenamiento completo")

##Evaluación, curvas y matriz de confusión

In [ ]:
# ── Evaluación en test set ────────────────────────────────────────────────────
preds_xlm_out = trainer_xlm.predict(tokenized_xlm["test"])
preds_xlm     = np.argmax(preds_xlm_out.predictions, axis=-1)
labels_test   = preds_xlm_out.label_ids

acc_xlm = accuracy_score(labels_test, preds_xlm)
f1_xlm  = f1_score(labels_test, preds_xlm, average="weighted", zero_division=0)

print(f"📊 XLM-RoBERTa Fine-tuning → Accuracy: {acc_xlm:.4f} | F1 weighted: {f1_xlm:.4f}")
print(classification_report(labels_test, preds_xlm, target_names=LABEL_NAMES, zero_division=0))

# Guardamos para la tabla final
resultados["Fine-tuning XLM-RoBERTa"] = {"accuracy": acc_xlm, "f1": f1_xlm}

# ── Curvas de entrenamiento ───────────────────────────────────────────────────
log_history_xlm = trainer_xlm.state.log_history
train_logs_xlm  = [l for l in log_history_xlm if "loss" in l and "eval_loss" not in l]
eval_logs_xlm   = [l for l in log_history_xlm if "eval_loss" in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([l["step"] for l in train_logs_xlm],
             [l["loss"] for l in train_logs_xlm],
             color="steelblue", alpha=0.7, label="Train loss")
axes[0].scatter([l["step"] for l in eval_logs_xlm],
                [l["eval_loss"] for l in eval_logs_xlm],
                color="tomato", s=80, zorder=5, label="Val loss")
axes[0].plot([l["step"] for l in eval_logs_xlm],
             [l["eval_loss"] for l in eval_logs_xlm],
             color="tomato", linestyle="--", alpha=0.6)
axes[0].set_title("Curva de pérdida – XLM-RoBERTa", fontweight="bold")
axes[0].set_xlabel("Pasos")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot([l["epoch"] for l in eval_logs_xlm],
             [l["eval_f1_weighted"] for l in eval_logs_xlm],
             color="darkorchid", marker="o", linewidth=2)
axes[1].set_title("F1 weighted por época – XLM-RoBERTa", fontweight="bold")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("F1 weighted")
axes[1].set_ylim(0, 1)
for l in eval_logs_xlm:
    axes[1].annotate(f"{l['eval_f1_weighted']:.3f}",
                     (l["epoch"], l["eval_f1_weighted"]),
                     textcoords="offset points", xytext=(0, 8), fontsize=9)

plt.suptitle("Entrenamiento XLM-RoBERTa – Documentos médicos en español",
             fontweight="bold")
plt.tight_layout()
plt.savefig("xlm_training.png", bbox_inches="tight")
plt.show()

# ── Matriz de confusión ───────────────────────────────────────────────────────
cm_xlm = confusion_matrix(labels_test, preds_xlm)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm_xlm, annot=True, fmt="d", cmap="Purples",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_title("Matriz de confusión – XLM-RoBERTa Fine-tuning",
             fontweight="bold", fontsize=13)
ax.set_ylabel("Etiqueta real")
ax.set_xlabel("Etiqueta predicha")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("xlm_confusion.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Tabla comparativa final ───────────────────────────────────────────────────
print("\n" + "="*65)
print("RESUMEN COMPARATIVO – CLASIFICACIÓN DE DOCUMENTOS MÉDICOS")
print("="*65)

tabla = {
    "Modelo":    ["Zero-Shot BART (inglés)",
                  "Zero-Shot BETO-XNLI (español)",
                  "Fine-tuning BETO",
                  "Fine-tuning XLM-RoBERTa"],
    "Parámetros":["406M", "110M", "110M", "278M"],
    "Idioma":    ["Inglés", "Español", "Español", "Multilingüe"],
    "Entrenamiento": ["Ninguno", "Ninguno", "4 épocas", "4 épocas"],
    "Accuracy":  [0.0600, 0.0500, 0.8063, 0.8063],
    "F1 weighted":[0.0262, 0.0378, 0.8086, 0.8111],
}

df_tabla = pd.DataFrame(tabla)
print(df_tabla.to_string(index=False))

# Gráfica comparativa de barras
fig, ax = plt.subplots(figsize=(12, 5))

modelos   = tabla["Modelo"]
accuracys = tabla["Accuracy"]
f1s       = tabla["F1 weighted"]
x         = np.arange(len(modelos))
width     = 0.35

bars1 = ax.bar(x - width/2, accuracys, width, label="Accuracy",
               color=["#d9534f","#d9534f","#3a86ff","#8338ec"], alpha=0.85)
bars2 = ax.bar(x + width/2, f1s, width, label="F1 weighted",
               color=["#d9534f","#d9534f","#3a86ff","#8338ec"], alpha=0.5)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(modelos, rotation=12, ha="right", fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Métrica")
ax.set_title("Comparación de modelos – Clasificación de documentos médicos en español",
             fontweight="bold")
ax.legend()
ax.axhline(y=1/16, color="gray", linestyle="--", alpha=0.5,
           label="Azar (1/16 clases)")
ax.text(3.6, 1/16 + 0.01, "azar", fontsize=8, color="gray")
plt.tight_layout()
plt.savefig("comparacion_final.png", bbox_inches="tight")
plt.show()

## Tabla Comparativa Final

| Modelo | Parámetros | Idioma | Entrenamiento | Accuracy | F1 weighted |
|--------|-----------|--------|---------------|----------|-------------|
| Zero-Shot BART (inglés) | 406M | Inglés | Ninguno | 6.00% | 0.026 |
| Zero-Shot BETO-XNLI (español) | 110M | Español | Ninguno | 5.00% | 0.038 |
| Fine-tuning BETO | 110M | Español | 4 épocas | **80.63%** | **0.809** |
| Fine-tuning XLM-RoBERTa | 278M | Multilingüe | 4 épocas | **80.63%** | **0.811** |

---

## Conclusiones

### 1. El fine-tuning es indispensable en dominios especializados

El hallazgo más contundente es el salto de ~6% (zero-shot) a ~81% (fine-tuning).
Ninguno de los modelos zero-shot pudo distinguir entre 16 tipos de documentos
médicos sin entrenamiento supervisado, independientemente del idioma. Esto confirma
que en dominios altamente especializados como el médico-legal, el vocabulario
técnico y la estructura del documento son señales que solo se aprenden con ejemplos.

### 2. El idioma del modelo importa en zero-shot, pero no en fine-tuning

En zero-shot, BETO-XNLI (español) obtuvo un F1 de 0.038 vs 0.026 de BART (inglés)
— una diferencia modesta pero en la dirección esperada. Sin embargo, en fine-tuning
BETO y XLM-RoBERTa convergen al mismo accuracy (80.63%), con XLM-RoBERTa apenas
superior en F1 (0.811 vs 0.809). La ventaja del modelo multilingüe desaparece
cuando hay suficientes datos de entrenamiento en el idioma objetivo.

### 3. Más parámetros no garantizan mejor resultado

XLM-RoBERTa tiene 2.5 veces más parámetros que BETO (278M vs 110M), tardó el
doble en entrenar, y obtuvo prácticamente el mismo resultado final. En la época 1,
BETO ya alcanzaba F1 de 0.752 mientras XLM-RoBERTa llegaba apenas a 0.547 —
el modelo especializado en español aprende el dominio más rápido. Para un caso
de uso en producción con recursos limitados, BETO sería la elección más eficiente.

### 4. La clase MEDICAL_RECORD es el talón de Aquiles de ambos modelos

Ambos modelos fallaron más en `MEDICAL_RECORD` (F1: 0.63 BETO, 0.59 XLM-RoBERTa).
La confusión ocurre principalmente con `OUTPATIENT_EXAMINATION` y
`DISCHARGE_SUMMARY` — documentos narrativos del historial clínico con vocabulario
muy solapado. Este es un problema estructural del dataset: estas clases comparten
secciones como antecedentes, diagnóstico y tratamiento, lo que hace difícil la
distinción incluso para un lector humano sin ver el encabezado del documento.

### 5. Reflexión sobre el dataset elegido

El dataset Meddies PII en español presentó un reto interesante y poco explorado:
clasificación de tipos de documentos clínicos, no de sentimiento ni intención.
La naturaleza especializada del dominio hizo que el zero-shot fallara completamente,
pero también hizo que el fine-tuning fuera especialmente efectivo — pasando de
azar absoluto a 81% de accuracy con solo 6.400 ejemplos de entrenamiento.

In [29]:
# Intentamos cargar sin token
from datasets import load_dataset

try:
    ds = load_dataset("Meddies/meddies-pii-cleaned-v1", "spanish")
    print("✅ Dataset público — no necesita token")
except Exception as e:
    print(f"❌ Necesita token: {e}")

✅ Dataset público — no necesita token
